In [1]:
import nibabel as nib
from totalsegmentator.python_api import totalsegmentator

In [2]:
input_img = nib.load("./ct.nii.gz")

In [5]:
input_img.shape

(511, 404, 339)

In [6]:
input_img = nib.load("./ct.nii.gz")
output_img = totalsegmentator(input=input_img)
nib.save(output_img, "./temp/")


If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Resampling...
  Resampled in 9.27s
Predicting part 1 of 5 ...


/home/hasan/research/visual_agent/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|███████████████████████████████████████████████████████████████████████████████████████████████| 36/36 [00:05<00:00,  7.06it/s]


Predicting part 2 of 5 ...


100%|███████████████████████████████████████████████████████████████████████████████████████████████| 36/36 [00:02<00:00, 15.49it/s]


Predicting part 3 of 5 ...


...

Predicting part 4 of 5 ...


100%|███████████████████████████████████████████████████████████████████████████████████████████████| 36/36 [00:02<00:00, 13.87it/s]


Predicting part 5 of 5 ...


100%|███████████████████████████████████████████████████████████████████████████████████████████████| 36/36 [00:02<00:00, 15.87it/s]


  Predicted in 82.62s
Resampling...


In [7]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

LABEL_NAMES = {
    1: "spleen", 2: "kidney_right", 3: "kidney_left", 4: "gallbladder",
    5: "liver", 6: "stomach", 7: "pancreas", 8: "adrenal_gland_right",
    9: "adrenal_gland_left", 10: "lung_upper_lobe_left", 11: "lung_lower_lobe_left",
    12: "lung_upper_lobe_right", 13: "lung_middle_lobe_right", 14: "lung_lower_lobe_right",
    15: "esophagus", 16: "trachea", 17: "thyroid_gland", 18: "small_bowel",
    19: "duodenum", 20: "colon", 21: "urinary_bladder", 22: "prostate",
    23: "kidney_cyst_left", 24: "kidney_cyst_right", 25: "sacrum",
    26: "vertebrae_S1", 27: "vertebrae_L5", 28: "vertebrae_L4", 29: "vertebrae_L3",
    30: "vertebrae_L2", 31: "vertebrae_L1", 32: "vertebrae_T12", 33: "vertebrae_T11",
    34: "vertebrae_T10", 35: "vertebrae_T9", 36: "vertebrae_T8", 37: "vertebrae_T7",
    38: "vertebrae_T6", 39: "vertebrae_T5", 40: "vertebrae_T4", 41: "vertebrae_T3",
    42: "vertebrae_T2", 43: "vertebrae_T1", 44: "vertebrae_C7", 45: "vertebrae_C6",
    46: "vertebrae_C5", 47: "vertebrae_C4", 48: "vertebrae_C3", 49: "vertebrae_C2",
    50: "vertebrae_C1", 51: "heart", 52: "aorta", 53: "pulmonary_vein",
    54: "brachiocephalic_trunk", 55: "subclavian_artery_right", 56: "subclavian_artery_left",
    57: "common_carotid_artery_right", 58: "common_carotid_artery_left",
    59: "brachiocephalic_vein_left", 60: "brachiocephalic_vein_right",
    61: "atrial_appendage_left", 62: "superior_vena_cava", 63: "inferior_vena_cava",
    64: "portal_vein_and_splenic_vein", 65: "iliac_artery_left", 66: "iliac_artery_right",
    67: "iliac_vena_left", 68: "iliac_vena_right", 69: "humerus_left", 70: "humerus_right",
    71: "scapula_left", 72: "scapula_right", 73: "clavicula_left", 74: "clavicula_right",
    75: "femur_left", 76: "femur_right", 77: "hip_left", 78: "hip_right",
    79: "spinal_cord", 80: "gluteus_maximus_left", 81: "gluteus_maximus_right",
    82: "gluteus_medius_left", 83: "gluteus_medius_right", 84: "gluteus_minimus_left",
    85: "gluteus_minimus_right", 86: "autochthon_left", 87: "autochthon_right",
    88: "iliopsoas_left", 89: "iliopsoas_right", 90: "brain", 91: "skull",
    92: "rib_left_1", 93: "rib_left_2", 94: "rib_left_3", 95: "rib_left_4",
    96: "rib_left_5", 97: "rib_left_6", 98: "rib_left_7", 99: "rib_left_8",
    100: "rib_left_9", 101: "rib_left_10", 102: "rib_left_11", 103: "rib_left_12",
    104: "rib_right_1", 105: "rib_right_2", 106: "rib_right_3", 107: "rib_right_4",
    108: "rib_right_5", 109: "rib_right_6", 110: "rib_right_7", 111: "rib_right_8",
    112: "rib_right_9", 113: "rib_right_10", 114: "rib_right_11", 115: "rib_right_12",
    116: "sternum", 117: "costal_cartilages"
}

ct_data  = nib.load('ct.nii.gz').get_fdata()
seg_data = nib.load('temp.nii').get_fdata().astype(np.uint8)

os.makedirs('frames', exist_ok=True)

num_labels = 117
cmap = plt.cm.get_cmap('tab20', num_labels + 1)

for i in range(ct_data.shape[2]):
    ct_slice  = ct_data[:, :, i]
    seg_slice = seg_data[:, :, i]

    present_labels = np.unique(seg_slice)
    present_labels = present_labels[present_labels > 0]

    if len(present_labels) == 0:
        continue

    # Two panels: scan on left, legend on right
    fig, (ax_scan, ax_legend) = plt.subplots(
        1, 2,
        figsize=(12, 8),
        gridspec_kw={'width_ratios': [3, 1]}
    )

    # --- Left: CT + overlay ---
    ax_scan.imshow(ct_slice.T, cmap='gray', origin='lower', vmin=-200, vmax=300)
    seg_rgba = cmap(seg_slice.T / num_labels)
    seg_rgba[..., 3] = np.where(seg_slice.T > 0, 0.45, 0.0)
    ax_scan.imshow(seg_rgba, origin='lower')
    ax_scan.axis('off')
    ax_scan.set_title(f'Slice {i}', fontsize=10, color='white')

    # --- Right: legend ---
    ax_legend.set_facecolor('black')
    ax_legend.axis('off')
    ax_legend.set_title('Segments', fontsize=9, color='white', pad=4)

    patches = [
        mpatches.Patch(
            color=cmap(label_id / num_labels),
            label=f"{int(label_id)}-{LABEL_NAMES.get(int(label_id), '?')}"
        )
        for label_id in sorted(present_labels)
    ]

    ax_legend.legend(
        handles=patches,
        loc='upper left',
        fontsize=7,
        framealpha=0,
        labelcolor='white',
        handlelength=1.2,
        handleheight=1.2,
        borderpad=0.5,
        labelspacing=0.4,
    )

    fig.patch.set_facecolor('black')
    plt.tight_layout(pad=0.5)
    plt.savefig(f'frames/slice_{i:04d}.png', dpi=150, bbox_inches='tight',
                facecolor='black')
    plt.close()

    if i % 20 == 0:
        print(f"  {i}/{ct_data.shape[2]} done...")

print("Done!")

/tmp/ipykernel_59539/333886352.py:49: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap('tab20', num_labels + 1)


  0/339 done...
  20/339 done...
  40/339 done...
  60/339 done...
  80/339 done...
  100/339 done...
  120/339 done...
  140/339 done...
  160/339 done...
  180/339 done...
  200/339 done...
  220/339 done...
  240/339 done...
  260/339 done...
  280/339 done...
  300/339 done...
  320/339 done...
Done!
